# 03. PEFT Orthogonal Finetuning: BOFT and OFT

- **Category:** 15_compression_peft_and_speculative_inference / 03_parameter_efficient_finetuning_peft
- **Lineage & Citations:** Qiu et al. (NeurIPS 2023, *Controlling Text-to-Image Diffusion by Orthogonal Finetuning*), Liu et al. (NeurIPS 2023, *BOFT: Butterfly Orthogonal Finetuning*).
- **Purpose:** Parameterize fine-tuning as hyperspherical rotations ($W = R W_0$ with $R^\top R = I$) via Cayley transform and butterfly factorization, provably preserving neuron angular relationships and preventing semantic collapse.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [PEFT SVD and Adaptive PiSSA Adalora Vera](02_peft_svd_and_adaptive_pissa_adalora_vera.ipynb) $\to$ `🔗 Downstream Syntheses:` [PEFT Adapters Ia3 Prefix Prompt and Llama Adapter](04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use OFT/BOFT when fine-tuning tasks require expanding representational rank or altering fundamental feature scales (where additive adapters like DoRA or LoRA+ excel). OFT strictly restricts updates to rotations.

## Mathematical Foundations & Governing Equations

$$
Q = \frac{1}{2}(M - M^\top), \quad R = (I - Q)(I + Q)^{-1}, \quad W_{\text{oft}} = R W_0, \quad R_{\text{total}} = R^{(2)} R^{(1)}
$$

### Asymptotic Time & Memory Complexity
- **Inference (Merged):** $\mathcal{O}(0)$ runtime & memory overhead (pre-multiply $W = R W_0$).
- **Forward Pass (Associative):** $\mathcal{O}(B \cdot d \cdot b)$ time, $\mathcal{O}(m \cdot b^2)$ parameters, avoiding dense $\mathcal{O}(d^2)$ allocations.

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/03_peft_orthogonal_finetuning_boft_and_oft.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def cayley_orthogonal(q_skew: torch.Tensor) -> torch.Tensor:
    b = q_skew.shape[0]
    eye = torch.eye(b, dtype=q_skew.dtype, device=q_skew.device)
    return torch.linalg.solve(eye + q_skew, eye - q_skew)

def block_oft_forward(x: torch.Tensor, w0: torch.Tensor, q_blocks: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    num_blocks, block_size, _ = q_blocks.shape
    r_blocks = [cayley_orthogonal(0.5 * (q_blocks[k] - q_blocks[k].T)) for k in range(num_blocks)]
    r_full = torch.block_diag(*r_blocks)
    return x @ (r_full @ w0), r_full

def butterfly_boft_forward(x: torch.Tensor, w0: torch.Tensor, q1: torch.Tensor, q2: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    _, r1 = block_oft_forward(x, w0, q1)
    _, r2 = block_oft_forward(x, w0, q2)
    r_total = r2 @ r1
    return x @ (r_total @ w0), r_total

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

r_c = cayley_orthogonal(weights["q_skew"])
assert float(torch.max(torch.abs(r_c - weights["r_cayley"]))) < 1e-5

oft_out, r_full = block_oft_forward(weights["x"], weights["w0"], weights["q_blocks1"])
assert float(torch.max(torch.abs(oft_out - weights["oft_out"]))) < 1e-5
assert float(torch.max(torch.abs(r_full - weights["r_full"]))) < 1e-5

boft_out, r_total = butterfly_boft_forward(weights["x"], weights["w0"], weights["q_blocks1"], weights["q_blocks2"])
assert float(torch.max(torch.abs(boft_out - weights["boft_out"]))) < 1e-5
assert float(torch.max(torch.abs(r_total - weights["r_total"]))) < 1e-5
print("All OFT & BOFT parity checks passed successfully with error < 1e-5!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: block_oft_forward(weights["x"], weights["w0"], weights["q_blocks1"]), iters=50, name="Block-OFT Forward"),
    benchmark_fn(lambda: butterfly_boft_forward(weights["x"], weights["w0"], weights["q_blocks1"], weights["q_blocks2"]), iters=50, name="Butterfly BOFT Forward"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: Additive Unconstrained Matrix vs Cayley Orthogonal Invariance
eye = torch.eye(4)
q = weights["q_skew"]
r_cayley = cayley_orthogonal(q)
ortho_err_cayley = float(torch.norm(r_cayley.T @ r_cayley - eye))

# Naive unconstrained additive update produces severe orthogonality violation
w_unconstrained = eye + q
ortho_err_naive = float(torch.norm(w_unconstrained.T @ w_unconstrained - eye))

print(f"Cayley Orthogonality Error ||R^T R - I||: {ortho_err_cayley:.2e}")
print(f"Naive Additive Error ||W^T W - I||:        {ortho_err_naive:.2e}")
assert ortho_err_cayley < 1e-5 and ortho_err_naive > 0.05, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Hyperspherical Angle Preservation:** OFT updates weights strictly along the orthogonal group $\mathbb{O}(d)$, preventing catastrophic forgetting by maintaining pairwise neuron cosine distances.
2. **Cayley Invariance:** The Cayley transform maps unconstrained skew-symmetric parameters to orthogonal matrices without costly matrix exponentials or Gram-Schmidt sweeps.
3. **Butterfly Interleaving:** BOFT breaks block-diagonal isolation via butterfly permutations, achieving global orthogonal interaction with logarithmic parameter overhead.

---
### Cross-Domain Generalisation Matrix
| Domain | Orthogonal / Rotation Analogue | Benefit of Orthogonal Invariance |
| :--- | :--- | :--- |
| **Generative Diffusion** | OFT subject-driven tuning | Preserves pre-trained compositionality and style |
| **LLM Safety Alignment** | Representation steering | Prevents jailbreak degradation of base safety boundary |
| **Robotics & Dynamics** | Lie-group $SO(3) / SE(3)$ control | Strict physical energy and norm conservation |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_peft_orthogonal_finetuning_boft_and_oft.md](../../../markdown/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/03_peft_orthogonal_finetuning_boft_and_oft.md)
- **PyTorch 2.x Module:** [03_peft_orthogonal_finetuning_boft_and_oft.py](../../../pytorch2/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/03_peft_orthogonal_finetuning_boft_and_oft.py)
- **Functional JAX Module:** [03_peft_orthogonal_finetuning_boft_and_oft.py](../../../jax/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/03_peft_orthogonal_finetuning_boft_and_oft.py)
- **Native Rust Candle Track:** [03_peft_orthogonal_finetuning_boft_and_oft.rs](../../../candle/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/03_peft_orthogonal_finetuning_boft_and_oft.rs)
- **Hardware Kernel (CPU / WGSL):** [03_peft_orthogonal_finetuning_boft_and_oft.rs](../../../kernels/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/03_peft_orthogonal_finetuning_boft_and_oft.rs)
- **Golden SafeTensors:** [03_peft_orthogonal_finetuning_boft_and_oft.safetensors](../../../weights/15_compression_peft_and_speculative_inference/03_parameter_efficient_finetuning_peft/03_peft_orthogonal_finetuning_boft_and_oft.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. PEFT SVD and Adaptive PiSSA Adalora Vera](02_peft_svd_and_adaptive_pissa_adalora_vera.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 04. PEFT Adapters Ia3 Prefix Prompt and Llama Adapter](04_peft_adapters_ia3_prefix_prompt_and_llama_adapter.ipynb) &rarr;